In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from facerec.distances import cosine_distances

In [ ]:
import numpy as np

def test_shape():
    a = np.random.rand(5, 128)
    b = np.random.rand(3, 128)
    assert cosine_distances(a, b).shape == (5, 3)


def test_self_distance_is_zero():
    a = np.random.rand(4, 128)
    d = cosine_distances(a, a)
    assert np.allclose(np.diag(d), 0)


def test_opposite_vectors():
    a = np.array([[1.0, 0.0], [0.0, 1.0]])
    d = cosine_distances(a, -a)
    assert np.allclose(np.diag(d), 2)


def test_orthogonal():
    a = np.array([[1.0, 0.0]])
    b = np.array([[0.0, 1.0]])
    assert np.allclose(cosine_distances(a, b), 1)


if __name__ == "__main__":
    test_shape()
    test_self_distance_is_zero()
    test_opposite_vectors()
    test_orthogonal()
    print("all good")

In [ ]:
from pathlib import Path
from skimage import io

known_faces_dir = Path.cwd().parent / "known_faces"
image_extensions = {".jpg", ".jpeg", ".png"}

def load_image(img_path):
    image = io.imread(str(img_path))
    if image.shape[-1] == 4:
        image = image[..., :3]  # drop alpha channel
    return image

# Expected layout: known_faces/<person_name>/*.jpg
dataset_images = {
    person_dir.name: [
        load_image(img_path)
        for img_path in sorted(person_dir.iterdir())
        if img_path.suffix.lower() in image_extensions
    ]
    for person_dir in sorted(known_faces_dir.iterdir())
    if person_dir.is_dir()
}


In [ ]:
from facenet_models import FacenetModel

model = FacenetModel()

detection_prob_threshold = 0.9  # min face-detection probability to keep a box

def get_descriptors(pic):
    boxes, probabilities, landmarks = model.detect(pic)
    if boxes is None:
        return []
    boxes = boxes[probabilities > detection_prob_threshold]
    if len(boxes) == 0:
        return []
    return list(model.compute_descriptors(pic, boxes))

# dataset: person_name -> list of shape-(512,) descriptor vectors
dataset = {
    person: [descriptor for pic in pics for descriptor in get_descriptors(pic)]
    for person, pics in dataset_images.items()
}


## TODO 3: Compute genuine vs. impostor distances

Using `cosine_distances` above:
- Genuine: each person's descriptors against their own (same-person pairs).
- Impostor: each pair of different people's descriptors against each other.

In [ ]:
genuine = []
impostor = []
people = list(dataset.keys())
#cosine distances between every pair of same people
for person in people:
    descriptors = np.array(dataset[person])
    if len(descriptors) < 2:
        continue
    distance_matrix = cosine_distances(descriptors, descriptors)  # shape (n, n)
    # Take the upper triangle (k=1 skips the diagonal, which
    #    is always 0 since a descriptor's distance to itself is 0) so each
    #    same-person pair is only counted once.
    iu = np.triu_indices(len(descriptors), k=1)
    genuine.extend(distance_matrix[iu])
#between every pair of different people's descriptor arrays.
for i, person_a in enumerate(people):
    for person_b in people[i + 1:]:
        distance_matrix = cosine_distances(np.array(dataset[person_a]), np.array(dataset[person_b]))
        impostor.extend(distance_matrix.ravel())

genuine = np.array(genuine)
impostor = np.array(impostor)


#The following are two different ways to determine the threshold#



Plot/compare genuine vs. impostor distances. Pick the threshold that best separates them (e.g. where false-accept rate and false-reject rate cross).

In [ ]:
# TODO: visualize genuine vs impostor distances and decide on `threshold`
import matplotlib.pyplot as plt
plt.hist(genuine, bins=50, alpha=0.5, label="Genuine")
plt.hist(impostor, bins=50, alpha=0.5, label="Impostor")
plt.xlabel("Distance")
plt.ylabel("Frequency")
plt.legend()
#find where false accept and false reject rate cross





For each person, compute the average descriptor vector across their samples, and also the
stdev of their descriptors. The stdev tells you how spread out a single person's own samples
are, so you can express "how far off" a candidate descriptor is in units of that person's own
variability (e.g. a z-score-like distance) rather than just a raw cosine distance. Compare
that against the same measure for impostors to help pick a threshold that accounts for
per-person variability.

In [ ]:
person_mean = []
person_std = []
for person in dataset:
    descriptors = np.array(dataset[person])
    person_mean = np.mean(descriptors, axis=0)
    person_std = np.std(descriptors, axis=0)
    person_mean[person] = person_mean
    person_std[person] = person_std
#now use person_mean and person_std to compute z-score-like distances for genuine and impostor samples, and analyze the distributions to refine threshold.
zscores=[]
for person in dataset:
    descriptors = np.array(dataset[person])
    person_mean = person_mean[person]
    person_std = person_std[person]
    # Compute z-score-like distances for genuine samples
    genuine_zscores = np.linalg.norm((descriptors - person_mean) / person_std, axis=1)
    zscores.extend(genuine_zscores)
    # Compute z-score-like distances for impostor samples
    for other_person in dataset:
        if other_person == person:
            continue
        other_descriptors = np.array(dataset[other_person])
        impostor_zscores = np.linalg.norm((other_descriptors - person_mean) / person_std, axis=1)
        zscores.extend(impostor_zscores)
